In [ ]:
import subprocess, sys, os, time, threading, glob, json, shutil, hashlib, zipfile
T_NB = time.time()
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode
sh(f"{sys.executable} -m pip install -q 'transformers==5.20.2' 'peft==0.21.2' accelerate 2>&1 | tail -3")
sh(f"{sys.executable} -m pip uninstall -y torchao 2>&1 | tail -1")
assert sh(f'''{sys.executable} -c "import transformers, peft, torch; print(transformers.__version__, peft.__version__, torch.__version__, torch.cuda.device_count(), torch.cuda.get_device_name(0)); assert (transformers.__version__, peft.__version__) == ('5.20.2', '0.21.2'); assert torch.cuda.device_count() == 2" ''') == 0, "DUNG: moi truong"
print("✅ moi truong", flush=True)


In [ ]:
W = "/kaggle/working"
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
D = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/sa3_mau.py", recursive=True) if "/notebooks/" not in p]
assert len(D) == 1, D; D = D[0]
for f, h in {'sa3_mau.py': '18d7aa8c48451851c7031202177bab04', 'grpo_spice.py': '07ea87b6d156d1faa391a4274dffd7cf', 'build_branch_data.py': '619e63e123a6dbf60086e65ee94a3912', 'train_v3.jsonl': 'e033ea809a18c0e5e3eb878305396f19', 'mau_0_cu.jsonl': '2fa5a6782ec16eb91986f49a05ebca50'}.items():
    assert md5(f"{D}/{f}") == h, f"DUNG: {f} lech md5"
for f in ("sa3_mau.py", "grpo_spice.py", "build_branch_data.py"):
    shutil.copyfile(f"{D}/{f}", f"{W}/{f}")
assert len(glob.glob(f"{D}/images/*.png")) == 1000
ENV = {**os.environ, "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1", "PYTORCH_ALLOC_CONF": "expandable_segments:True"}
MERGED = f"{W}/s1_merged"
r = subprocess.run(["python", "grpo_spice.py", "--merge", "--bundle", D, "--adapter", f"{D}/adapter_s1_seed101", "--merged", MERGED], cwd=W, capture_output=True, text=True, env=ENV)
print((r.stdout + r.stderr)[-800:], flush=True); assert r.returncode == 0, "DUNG: hoa S1 loi"
CAT = 3600 * 3.0
def chay(cmd, log, gpu):
    t0 = time.time()
    with open(log, "a") as f:
        q = subprocess.Popen(cmd, cwd=W, stdout=f, stderr=subprocess.STDOUT, env={**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)})
        while q.poll() is None:
            time.sleep(120)
            L = [l.strip() for l in open(log, errors="ignore") if l.strip()]
            print(f"{(time.time()-T_NB)/3600:5.2f} h · gpu{gpu} · {os.path.basename(log)} · {L[-1][:120] if L else 'chua co dong'}", flush=True)
            if time.time() - T_NB > CAT:
                print("⛔ het gio, dung de Output kip luu", flush=True); q.terminate(); q.wait(); return None
    print(f"--- {os.path.basename(log)} · ma thoat {q.returncode} · {(time.time()-t0)/3600:.2f} h", flush=True)
    if q.returncode != 0:
        print(open(log, errors="ignore").read()[-2500:], flush=True)
    return q.returncode
ARGS = ["--data", D, "--merged", MERGED, "--sft", f"{D}/sft_final"]


In [ ]:
# chạy tiếp: mỗi phần bắt đầu từ 156 dòng cũ của shard 0 ⇒ không sinh lại
for i in (0, 1):
    shutil.copyfile(f"{D}/mau_0_cu.jsonl", f"{W}/mau_0_p{i}.jsonl")
print("dong cu", sum(1 for _ in open(f"{W}/mau_0_p0.jsonl")), flush=True)


In [ ]:
RC = {}
def phan(i):
    RC[i] = chay(["python", "-u", "sa3_mau.py", *ARGS, "--shard", "0", "--nshard", "2", "--part", str(i), "--npart", "2", "--out", f"{W}/mau_0_p{i}.jsonl"], f"{W}/mau_0_p{i}.log", i)
th = [threading.Thread(target=phan, args=(i,)) for i in (0, 1)]
[t.start() for t in th]; [t.join() for t in th]
for i in (0, 1):
    n = sum(1 for _ in open(f"{W}/mau_0_p{i}.jsonl"))
    print(f"phan {i}: ma thoat {RC.get(i)} · {n} dong (156 cu + ~172 moi)", flush=True)
    print(json.load(open(f"{W}/mau_0_p{i}.jsonl.probe.json")), flush=True)


In [ ]:
with zipfile.ZipFile(f"{W}/sa3_mau_out.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{W}/*.jsonl") + glob.glob(f"{W}/*.probe.json") + glob.glob(f"{W}/*.log"):
        z.write(p, os.path.basename(p))
print("zip", os.path.getsize(f"{W}/sa3_mau_out.zip") // 1024, "KB", flush=True)
shutil.rmtree(f"{W}/s1_merged", ignore_errors=True)
